# Other embedders

In [1]:
import lancedb 
import pandas as pd

db = lancedb.connect("knowledge_base")

df = pd.read_json("data/jokes.json")
df.head()

,jokes
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


## Sentence transformer

In [2]:
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector

model = (
    get_registry()
    .get("sentence-transformers")
    .create(name="intfloat/multilingual-e5-small", device="cpu")
)

class JokeModel(LanceModel):
    jokes: str = model.SourceField()
    vector: Vector(model.ndims()) = model.VectorField() 

print(f"{model.ndims() = }")

table = db.create_table("jokes_sentence_transformers", schema=JokeModel)
table


c:\Users\anjag\Documents\github\AIOps\AIOps_course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2386.56it/s]


model.ndims() = 384


LanceTable(name='jokes_sentence_transformers', _conn=LanceDBConnection(uri='c:\\Users\\anjag\\Documents\\github\\AIOps\\AIOps_course\\code_along\\class\\09_lancedb\\knowledge_base'))

In [3]:
table.add(df)

AddResult(version=2)

In [4]:
table.search().limit(5).to_pandas()

,jokes,vector
0,Parallel lines have so much in common—it’s sad...,"[0.037198, -0.019918, -0.051841, -0.032693, 0...."
1,"ETL stands for “Extract, Transform, Leave for ...","[0.050497, -0.006426, -0.071011, -0.027726, 0...."
2,What do you call a snake that runs your script...,"[0.073168, -0.00455, -0.027777, -0.054148, 0.0..."
3,"Gold walks into a bar. The bartender says, “Au...","[0.045366, -0.05297, -0.066901, -0.02893, 0.11..."
4,C# devs don’t argue; they just throw exceptions.,"[0.028011, 0.004646, -0.057664, -0.064801, 0.0..."


In [5]:
table.search("math jokes").to_pandas().head()

,jokes,vector,_distance
0,My math teacher said I’m average… how mean!,"[-0.000705, 0.014578, -0.084753, -0.031266, 0....",0.290241
1,"Dear algebra, please stop asking us to find yo...","[0.05175, 0.00825, -0.020384, -0.038882, 0.084...",0.300916
2,Why’s 6 afraid of 7? Because 7 8 9.,"[0.044206, -0.02136, -0.043387, -0.028015, 0.0...",0.312815
3,The C# compiler walked into a bar. The bartend...,"[0.02272, -0.017781, -0.066756, -0.017234, 0.1...",0.339917
4,I told a chemistry joke… there was no reaction.,"[0.048405, 0.011754, -0.043688, -0.029928, 0.1...",0.347602


In [6]:
table.search("mattematikskämt").to_pandas().head()

,jokes,vector,_distance
0,My math teacher said I’m average… how mean!,"[-0.000705, 0.014578, -0.084753, -0.031266, 0....",0.400908
1,Why did the chemist ground his kids? Because t...,"[-0.000735, -0.029147, -0.051178, -0.053038, 0...",0.423457
2,Parallel lines have so much in common—it’s sad...,"[0.037198, -0.019918, -0.051841, -0.032693, 0....",0.432981
3,Why’s 6 afraid of 7? Because 7 8 9.,"[0.044206, -0.02136, -0.043387, -0.028015, 0.0...",0.440617
4,Python is so friendly even whitespace gets a say.,"[0.020338, 0.010201, -0.026083, -0.032346, 0.0...",0.444238


In [7]:
from custom_embedders import nvidia_nemotron
from lancedb.pydantic import LanceModel, Vector

model = get_registry().get("nvidia-nemotron").create()

class JokeModel(LanceModel):
    jokes: str = model.SourceField()
    vector: Vector(model.ndims()) = model.VectorField() 

model.ndims()

4096

In [8]:
table = db.create_table("jokes_nvidia", schema=JokeModel)

table.add(df)

AddResult(version=2)

In [9]:
table.search("programmerare tycker inte om ljus").limit(5).to_pandas()

,jokes,vector,_distance
0,C# programmers love coffee—they're always work...,"[-0.00407, -0.055267, 0.009743, 0.003815, 0.00...",1.439124
1,Why did the Python programmer get bitten? Beca...,"[-0.013878, -0.042084, 0.011299, -0.019653, 0....",1.522183
2,Why do data engineers hate nature? Too many un...,"[-0.006134, -0.076904, 0.011269, -0.00084, 0.0...",1.526139
3,C# devs don’t argue; they just throw exceptions.,"[0.009048, -0.062347, 0.02359, 0.017288, 0.013...",1.539856
4,Why is C# like a musical? It has so many classes.,"[-0.00077, -0.090088, 0.015549, 0.007221, 0.01...",1.578951
